# 08 — Phân tích lỗi CNN và hoàn thiện báo cáo

Phân tích dự đoán test đã lưu, minh họa Grad-CAM và so sánh với baseline KNN/SVM
trên cùng manifest. Không điều chỉnh trọng số hoặc cấu hình sau khi xem lỗi test.
Cuối notebook dựng báo cáo từ số đo thật và đóng gói toàn bộ kết quả.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA bằng các mức thư mục giới hạn.
from pathlib import Path
import os, sys

kaggle_input = Path('/kaggle/input')
mounts = []
if kaggle_input.is_dir():
    mounts = [p for pattern in ('datasets/*/*', '*')
              for p in kaggle_input.glob(pattern) if p.is_dir()]
project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *mounts,
                       *(child for mount in mounts for child in mount.glob('*') if child.is_dir())])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/CNN/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError('Gắn đúng một Dataset code có src/CNN/pipeline.py; '
                            f'có thể đặt CS231_PROJECT_ROOT. Tìm thấy: {project_matches}')
PROJECT_ROOT = project_matches[0]
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [PROJECT_ROOT / 'DATA', *(p / 'DATA' for p in mounts), *mounts])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir() and (p / 'dev/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError('Gắn đúng Dataset DATA có train/dev; '
                            f'có thể đặt CS231_DATA_ROOT. Tìm thấy: {data_matches}')
DATA_ROOT = data_matches[0]
result_hint = os.environ.get('CS231_CNN_RESULTS_ROOT')
RESULTS_ROOT = (Path(result_hint) if result_hint else
                Path('/kaggle/working/CS231_CNN/results_224') if Path('/kaggle/working').is_dir()
                else PROJECT_ROOT / 'src/CNN/results_224').resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['CS231_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['CS231_DATA_ROOT'] = str(DATA_ROOT)
print('Code:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
from src.CNN.runtime import restore_previous_output
restore_previous_output(RESULTS_ROOT, '07', 'test_final/metrics_test.json')


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Image, Markdown
from src.CNN.data import CLASSES, assert_protocol
from src.CNN.runtime import mark_stage
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())


## 1. Lỗi theo lớp và chênh lệch VND

Đọc lại kết quả test đã kiểm tra hash. Tổng lỗi, tổng chênh lệch và bảng cặp nhầm
được tạo từ cùng dự đoán, không phải từ một tập ảnh chọn riêng.


In [ ]:
from src.CNN.analysis import analyze_errors, compare_baselines
cases, directed_errors = analyze_errors(DATA_ROOT, RESULTS_ROOT, PROJECT_ROOT)
display(directed_errors.sort_values('total_gap_vnd', ascending=False))
display(pd.read_csv(RESULTS_ROOT / 'error_analysis/error_pairs_bidirectional.csv'))
display(cases[['case_id', 'y_true_vnd', 'y_pred_vnd', 'top_softmax', 'softmax_gap', 'selection_reason']])


## 2. Ảnh đối chứng và Grad-CAM

Tối đa12 ảnh không trùng được chọn theo cặp lỗi phổ biến, tổng chênh lệch lớn,
điểm softmax cao/sát nhau và ảnh đúng đối chứng. Grad-CAM dùng ReLU thứ hai của
khối cuối, trước MaxPool; mỗi heatmap chuẩn hóa riêng theo cực đại, map0 được ghi rõ.
Heatmap gợi vùng đáp ứng của logit, không chứng minh yếu tố gây lỗi. Softmax cao vẫn có thể sai.


In [ ]:
for path in sorted((RESULTS_ROOT / 'error_analysis').glob('cases_page_*.png')):
    display(Image(filename=str(path)))
display(pd.read_csv(RESULTS_ROOT / 'error_analysis/gradcam_checks.csv'))
display(pd.read_csv(RESULTS_ROOT / 'error_analysis/visual_review_notes.csv').fillna(''))


## 3. Đối chiếu KNN và SVM

Baseline được sao chép có nguồn commit và SHA-256. Chỉ so khi train/dev/test có
cùng đường dẫn, nhãn và thứ tự ảnh; tính lại metric baseline từ CSV trước khi ghép cặp.
CNN có đặc trưng học được và có thể augmentation, nên bảng đánh giá pipeline hoàn chỉnh,
chưa tách riêng tác động thuật toán hoặc điều kiện huấn luyện.


In [ ]:
metrics_comparison, paired = compare_baselines(
    RESULTS_ROOT, PROJECT_ROOT / 'src/CNN/reference_results')
display(metrics_comparison)
display(paired)
display(pd.read_csv(RESULTS_ROOT / 'comparison/per_class_comparison_test.csv'))


## 4. Dựng báo cáo theo bằng chứng đã lưu

Script ghép phần phương pháp với bảng dev/test, cặp lỗi, so sánh và các hình thật.
Metric được tính lại từ dự đoán; không sinh số giả hoặc suy nguyên nhân thị giác từ score.
Ghi chú quan sát cần được hoàn thiện sau khi xem ảnh và giữ tách khỏi giả thuyết.


In [ ]:
import runpy
report_script = PROJECT_ROOT / 'docs/CNN_REPORT/build_report_224.py'
if not report_script.is_file():
    raise FileNotFoundError('Dataset code cần có docs/CNN_REPORT/ để dựng báo cáo.')
report_module = runpy.run_path(str(report_script))
REPORT_ROOT = RESULTS_ROOT / 'report'
report_path = report_module['build_report'](RESULTS_ROOT, REPORT_ROOT, PROJECT_ROOT)
print('Báo cáo:', report_path)
display(Markdown(report_path.read_text(encoding='utf-8')))


## 5. Đóng gói kết quả tích lũy

ZIP chứa manifest, lịch sử, checkpoint, dự đoán, metric, hình và báo cáo.
Bản đầy đủ cần cho việc tái sử dụng mô hình; Git chỉ giữ các đầu ra gọn,
không thay checkpoint bằng một bảng CSV khi cần suy luận hoặc Grad-CAM.


In [ ]:
import shutil
mark_stage(RESULTS_ROOT, '08')
archive = shutil.make_archive(str(RESULTS_ROOT.parent / 'CS231_CNN_RESULTS'), 'zip',
                              root_dir=RESULTS_ROOT.parent, base_dir=RESULTS_ROOT.name)
print('ZIP kết quả:', archive)
